# Healthcare Test Results Classification
### Multi-Class Clinical Prediction using Demographic and Medical Records

**Dataset:** Kaggle Healthcare Dataset (`prasad22/healthcare-dataset`)
**Objective:** Predict clinical test results (`Normal`, `Abnormal`, `Inconclusive`) based on patient demographics, medical condition, admission type, medication, and billing.
**Key Techniques:**
- Comprehensive Exploratory Data Analysis (EDA) & Clinical Profiling
- Feature Preprocessing: `ColumnTransformer`, `OneHotEncoder`, `StandardScaler`
- Multi-Model Comparison: Logistic Regression, Random Forest, XGBoost
- Bayesian Hyperparameter Optimization with **Optuna**
- Comprehensive Evaluation: Stratified F1-Macro, Multi-Class Confusion Matrix, ROC-AUC

In [ ]:
# 1. Environment Setup & Dependency Installation
!pip install -q xgboost optuna kagglehub

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from xgboost import XGBClassifier
import optuna
import kagglehub

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")

In [ ]:
# 2. Data Ingestion
try:
    path = kagglehub.dataset_download("prasad22/healthcare-dataset")
    import os
    csv_files = [f for f in os.listdir(path) if f.endswith(".csv")]
    df = pd.read_csv(os.path.join(path, csv_files[0]))
    print(f"Dataset successfully loaded from KaggleHub: {df.shape}")
except Exception as e:
    print("Loading synthetic clinical dataset for offline demonstration:", e)
    np.random.seed(42)
    n_samples = 2500
    df = pd.DataFrame({
        "Name": [f"Patient_{i}" for i in range(n_samples)],
        "Age": np.random.randint(18, 85, n_samples),
        "Gender": np.random.choice(["Male", "Female"], n_samples),
        "Blood Type": np.random.choice(["A+", "A-", "B+", "B-", "AB+", "AB-", "O+", "O-"], n_samples),
        "Medical Condition": np.random.choice(["Diabetes", "Hypertension", "Asthma", "Arthritis", "Cancer", "Obesity"], n_samples),
        "Date of Admission": pd.date_range("2023-01-01", periods=n_samples, freq="h"),
        "Doctor": [f"Dr. {name}" for name in np.random.choice(["Smith", "Patel", "Johnson", "Williams"], n_samples)],
        "Hospital": np.random.choice(["General Hospital", "City Medical", "St. Jude", "Mayo Clinic"], n_samples),
        "Insurance Provider": np.random.choice(["Blue Cross", "Medicare", "Aetna", "UnitedHealthcare", "Cigna"], n_samples),
        "Billing Amount": np.round(np.random.exponential(12000, n_samples) + 1500, 2),
        "Room Number": np.random.randint(100, 500, n_samples),
        "Admission Type": np.random.choice(["Emergency", "Elective", "Urgent"], n_samples),
        "Discharge Date": pd.date_range("2023-01-05", periods=n_samples, freq="h"),
        "Medication": np.random.choice(["Aspirin", "Ibuprofen", "Penicillin", "Paracetamol", "Lipitor"], n_samples),
        "Test Results": np.random.choice(["Normal", "Abnormal", "Inconclusive"], n_samples, p=[0.35, 0.40, 0.25])
    })

df.head()

In [ ]:
# 3. Data Cleansing & Feature Engineering
# Drop non-predictive identifiers and administrative attributes
drop_cols = ["Name", "Doctor", "Hospital", "Room Number"]
df_clean = df.drop(columns=[col for col in drop_cols if col in df.columns]).copy()

# Calculate Length of Stay (in days) from admission & discharge timestamps
if "Date of Admission" in df_clean.columns and "Discharge Date" in df_clean.columns:
    df_clean["Date of Admission"] = pd.to_datetime(df_clean["Date of Admission"])
    df_clean["Discharge Date"] = pd.to_datetime(df_clean["Discharge Date"])
    df_clean["Length of Stay"] = (df_clean["Discharge Date"] - df_clean["Date of Admission"]).dt.total_seconds() / (24 * 3600)
    df_clean.drop(columns=["Date of Admission", "Discharge Date"], inplace=True)

print("Cleaned Features:", df_clean.columns.tolist())
print("Target Distribution:\n", df_clean["Test Results"].value_counts(normalize=True))

In [ ]:
# 4. Exploratory Data Analysis & Visualizations
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.countplot(data=df_clean, x="Test Results", palette="Set2", ax=axes[0])
axes[0].set_title("Distribution of Test Results Target", fontsize=13)

sns.boxplot(data=df_clean, x="Test Results", y="Billing Amount", palette="Set3", ax=axes[1])
axes[1].set_title("Billing Amount by Test Result", fontsize=13)

pd.crosstab(df_clean["Medical Condition"], df_clean["Test Results"], normalize="index").plot(
    kind="bar", stacked=True, colormap="viridis", ax=axes[2]
)
axes[2].set_title("Medical Condition vs Test Results Ratio", fontsize=13)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# 5. Preprocessing Pipeline & Feature Transformation
X = df_clean.drop(columns=["Test Results"])
y_raw = df_clean["Test Results"]

le = LabelEncoder()
y = le.fit_transform(y_raw)
class_names = le.classes_
print("Target Classes:", dict(zip(range(len(class_names)), class_names)))

num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)
    ]
)

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train = preprocessor.fit_transform(X_train_raw)
X_test = preprocessor.transform(X_test_raw)
print(f"Preprocessed Feature Matrix: Train shape {X_train.shape}, Test shape {X_test.shape}")

In [ ]:
# 6. Multi-Model Benchmark Comparison
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost": XGBClassifier(eval_metric="mlogloss", random_state=42)
}

benchmark_results = []
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, clf in models.items():
    scores = cross_val_score(clf, X_train, y_train, cv=cv, scoring="f1_macro")
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    test_f1 = f1_score(y_test, y_pred, average="macro")
    test_acc = accuracy_score(y_test, y_pred)
    benchmark_results.append({
        "Model": name,
        "CV F1-Macro": scores.mean(),
        "CV Std": scores.std(),
        "Test F1-Macro": test_f1,
        "Test Accuracy": test_acc
    })

results_df = pd.DataFrame(benchmark_results)
print("=== Model Benchmark Results ===")
results_df

In [ ]:
# 7. Hyperparameter Optimization with Optuna
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 200),
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "eval_metric": "mlogloss",
        "random_state": 42
    }
    model = XGBClassifier(**params)
    scores = cross_val_score(model, X_train, y_train, cv=3, scoring="f1_macro")
    return scores.mean()

optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=15, timeout=120)

print(f"Optuna Best Trial Score (CV Macro F1): {study.best_value:.4f}")
print("Best Hyperparameters:", study.best_params)

In [ ]:
# 8. Final Model Training & Multi-Class Evaluation
best_xgb = XGBClassifier(**study.best_params, eval_metric="mlogloss", random_state=42)
best_xgb.fit(X_train, y_train)
y_pred_opt = best_xgb.predict(X_test)

print("=== Final Classification Report ===")
print(classification_report(y_test, y_pred_opt, target_names=class_names))

cm = confusion_matrix(y_test, y_pred_opt)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.title("Confusion Matrix - Optimized XGBoost")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()